# Test data_pipeline_nap

In [1]:
from approach_nap.data_pipeline_nap import build_nap_dataloaders 

log_path = "baselines/next_activity_prediction/evaluation/raw_datasets_that_are_not_evaluated/SEPSIS.xes.gz"

train_loader, val_loader, test_loader, act_vocab = build_nap_dataloaders(log_path)

batch = next(iter(train_loader))
print(f"\nSample batch – nodes: {batch.x.shape}, "
      f"edges: {batch.edge_index.shape[1]}, "
      f"labels: {batch.y.shape}")
print(f"Activity vocab: {list(act_vocab.items())[:5]} …")

Loading event log …


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

Cases  – train: 607  val: 152  test: 210
Vocabulary size (train+val): 16


Test graphs : 100%|██████████████████████████| 210/210 [00:00<00:00, 438.27it/s]

Graphs – train: 8149  val: 1780  test: 2728

Sample batch – nodes: torch.Size([379, 16]), edges: 623, labels: torch.Size([32, 16])
Activity vocab: [('Admission IC', 0), ('Admission NC', 1), ('CRP', 2), ('ER Registration', 3), ('ER Sepsis Triage', 4)] …


# Run and evaluate POG-NAP

In [2]:
from approach_nap.run_nap import run 

log_path = "baselines/next_activity_prediction/evaluation/raw_datasets_that_are_not_evaluated/SEPSIS.xes.gz"
log_name = "SEPSIS"
results_dir = "pog_results"
run(log_path, log_name, results_dir)


Log : SEPSIS
Device : cuda
Loading event log …


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

Cases  – train: 607  val: 152  test: 210
Vocabulary size (train+val): 16


Test graphs : 100%|██████████████████████████| 210/210 [00:00<00:00, 631.76it/s]


Graphs – train: 8149  val: 1780  test: 2728

Model parameters: 55,696
Epoch   1  train_loss=1.8932  val_loss=1.6179  val_acc=0.3522  val_f1=0.2251
Epoch   2  train_loss=1.5565  val_loss=1.4916  val_acc=0.4253  val_f1=0.3270
Epoch   3  train_loss=1.4868  val_loss=1.4215  val_acc=0.4298  val_f1=0.3276
Epoch   4  train_loss=1.4471  val_loss=1.3739  val_acc=0.4551  val_f1=0.3780
Epoch   5  train_loss=1.4051  val_loss=1.3471  val_acc=0.4781  val_f1=0.4147
Epoch   6  train_loss=1.3698  val_loss=1.3025  val_acc=0.5522  val_f1=0.5100
Epoch   7  train_loss=1.3317  val_loss=1.2499  val_acc=0.5287  val_f1=0.4895
Epoch   8  train_loss=1.2945  val_loss=1.2077  val_acc=0.5410  val_f1=0.4931
Epoch   9  train_loss=1.2462  val_loss=1.2015  val_acc=0.5629  val_f1=0.5244
Epoch  10  train_loss=1.1984  val_loss=1.1030  val_acc=0.6129  val_f1=0.5777
Epoch  11  train_loss=1.1630  val_loss=1.0760  val_acc=0.6242  val_f1=0.5949
Epoch  12  train_loss=1.1340  val_loss=1.0372  val_acc=0.6213  val_f1=0.5979
Epoch 

{'accuracy': 0.6532258064516129, 'f1': 0.6314323024293653}

# Compare with baselines

In [3]:
from baselines.next_activity_prediction.results_collector import get_results_for_log
import pandas as pd

log = "SEPSIS"

results_nap = pd.concat([
    pd.read_csv("pog_results/results_nap_gnn.csv")
      .query("log == @log")[["model", "method", "accuracy", "f1"]],
    get_results_for_log(log)[["model", "method", "accuracy", "f1"]]
], ignore_index=True)
results_nap

,model,method,accuracy,f1
0,GNN,nap,0.653226,0.631432
1,everman,Activity-Context Bag Of Words PPMI w_5,0.597345,0.570349
2,everman,Activity-Context N-Grams PPMI CBOW w_5,0.582369,0.554189
3,everman,Bose 2009 Substitution Scores,0.543907,0.493423
4,everman,De Koninck 2018 act2vec CBOW w_3,0.541865,0.485156
5,everman,Gamallo Fernandez 2023 Context Based w_3,0.545609,0.493565
6,everman,one_hot,0.339687,0.245328
7,pydream,NAP,0.557188,0.534609
8,tax,one_hot,0.638530,0.628292


## Important notes

All models are evaluated under identical experimental conditions: the same temporal train/val/test split, the same activity vocabulary, and one prediction per event. The proposed GNN approach differs only in how the prefix is represented, as a partial-order DAG rather than a flat sequence.